# Counting operations
By working through the last notebooks you will have gained a good understanding of the basics of the Python programming language. By now, you should be able to implement simple algorithms to answer mathematical questions. The problems that are solved in many real-life applications are often large. For example we might need to invert a large matrix or evaluate a high-degree polynomial. In these cases it is important to get a good understanding of how the runtime of our code will grow with the problem size, i.e. the matrix size and polynomial degree. In this notebook we will start to develop the tools for answering this question by looking at a simple example. We will see that for a given mathematical problem there can be more than one correct implementation, and that the computational cost of different implementations can differ significantly.
<hr style="height: 2px">

### What you will learn
In this notebook we will cover the following topics:

* Polynomial evaluation
* Estimating runtime
* Counting operations
* Horner's rule
* Aymptotic growth of the computational cost

<hr style="height: 2px">

*&#169; Eike Mueller, University of Bath 2019-2023. These lecture notes are copyright of Eike Mueller, University of Bath. They are provided exclusively for educational purposes at the University and are to be downloaded or copied for your private study only. Further distribution, e.g. by upload to external repositories, is prohibited.*

## Polynomial evaluation
Imagine that we want to evaluate the following general polynomial of degree $n$

$$
p(x) = a_0 + a_1\cdot x +a_2\cdot x^2 + a_3\cdot x^2 + \dots + a_n\cdot x^n = \sum_{j=0}^{n} a_j\cdot x^j
$$

For simplicity we assume that both the value $x$ and the coefficients $a_0,a_1,a_2,\dots,a_n$ are integer valued. Here is some simply pseudocode for evaluating the polynomial $p(x)$:

### Algorithm: Naive polynomial evaluation
**Input**: List $a=[a_0,a_1,\dots,a_n]$ of coefficients, value $x$ **Output**: Value of the polynomial $p(x)=\sum_{j=0}^{n} a_j x^j$
1. $p\mapsto 0$
1. **for** $j=0,1,2,\dots,n$ **do**
1. $~~~~$ $q\mapsto a_j$
1. $~~~~$ **for** $k=0,1,2,\dots,j-1$ **do**
1. $~~~~~~~~$ $q\mapsto x\cdot q$
1. $~~~~$ **end for**
1. $~~~~$ $p\mapsto p+q$
1. **end for**
1. **return** $p$

The inner for-loop over $k$ computes $q=a_j\cdot x^j$ by repeatedly multiplying $a_j$ with $x$. A Python function, which gets passed a list of coefficient `a_coeff` and a value `x` at which to evaluate the polynomial is readily written down:

In [ ]:
def poly_naive(a_coeff,x):
    '''Evaluate polynomial p(x) = sum_{j=0}^{n} a_j*x^j
    
    input:
      * a_coeff: List with polynomial cofficients a_0,a_1, ..., a_n
      * x: value at which to evaluate the polynomial
    output:
      value of polynomial p(x)
    '''
    degree = len(a_coeff)-1     # The list a_coeff contains n+1 elements, where n is the degree 
    poly = a_coeff[0]    
    for j in range(1,degree+1):   # Loop from 1 to n
        aj_xj = a_coeff[j]
        for k in range(j):
            aj_xj *= x
        poly += aj_xj # Update polynomial
    return poly

Let us use this function to evaluate the third-degree polynomial $2+3\cdot x+4\cdot x^2+5\cdot x^3$ for $x=6$:

In [ ]:
a_coeff = [2,3,4,5]
x = 6
poly_naive(a_coeff,x)

(To assure yourself that the code is working as expected, you might want to use the function `poly_naive` to evaluate simpler polynomials such as $1+4x$ or $1+2x+x^3$).

### Estimating runtimes
As mentioned above, we might be interested in the case where the polynomial degree $n$ is very large. Can we estimate the runtime of the code for a given value of $n$? You might object that this is a futile exercise since the runtime will depend on the computer that the code is run on and that is correct. However, it turns out that while the runtime is machine dependent, it is still possible to do the following:

Imagine that for a given value of $n$ we can count the number of elementary arithmetic operations (short ''elops''), i.e. the number of additions, multiplications, subtractions and divisions in our algorithm. Let us call this number of operations $E(n)$. Assuming that executing one of those elops takes some time $\tau_{\text{computer}}$ on a particular computer, we can then express the runtime $T(n)$ as

$$
T(n) = \tau_{\text{computer}} E(n)
$$

Even though the runtime is machine dependent, this is an extremely helpful expression, since this machine dependence is absorbed into the factor $\tau_{\text{computer}}$, which is independent of $n$. For example, we can use it the expression above to predict how the runtime increases if the problem size $n$ doubles:

$$
T(2n) = \frac{E(2n)}{E(n)}T(n)
$$

Hence, knowledge of the number of elops $E(n)$ allows us to say something about the *growth* of the runtime, i.e. the ratio $T(2n)/T(n)$.

### Counting operations
How can we compute $E(n)$? Well, all we need to do is look at the algorithm and work out the number of elops. Looking at the pseudocode above, first observe that we need to do one addition in each of the $n$ iterations in the for loop, so that is $n$ additions in total. Next, let us look at the multiplications. To raise a number $x$ to the $j$-th power we need to do $j-1$ multiplications, since

$$
x^j = \underbrace{x\cdot x\cdot x \cdots x}_{j\;\text{factors}}
$$

Hence, the number of multiplications to compute $x^j$ for all $j=1,2,\dots,n$ is

$$
0 + 1 + 2 + 3 + \dots + n-1 = \sum_{j=0}^{n-1} = \frac{n(n-1)}{2}
$$

We also have to multiply $a_j$ by $x^j$ for each $j=1,2,\dots,n$, so that is another $n$ multiplications.

We conclude that the total number of elops is

$$
E_{\text{naive}}(n) = \underbrace{n}_{\text{additions}} + \underbrace{\frac{n(n-1)}{2} + n}_{\text{multiplications}} = \frac{1}{2}n^2 + \frac{3}{2}n
$$

As an example, let's look at a polynomial of degree four, which can be evaluated with 4 additions and 10 multiplications:

$$
p(x) = a_0 + a_1\cdot x + a_2\cdot x\cdot x + a_3 \cdot x\cdot x\cdot x + a_4 \cdot x\cdot x\cdot x\cdot x
$$

You might object that in the above Python code  we could have replaced the lines

```Python
q = a_coeff[j]
for k in range(j):
    q *= x
```

by

```Python
q = q_coeff[j]*x**j
```

since the double-star operator `**` raises a number to some power. However, this would not necessarily reduce the number of operations since ''under the hood'' Python still has to multiply $x$ by itself $j-1$ times to compute $x^j$ (we will, however, later look at a method for computing $x^j$ with fewer than $j-1$ operations for large values of $j$).

## Horner's rule
It turns out that there is a more efficient way of evaluating the polynomial $p(x)$. For this, rewrite the polynomial as follows:

$$
\begin{aligned}
p(x) &= a_0 + a_1x + a_2x^2 + a_3x^3 +  \dots + a_n x^n\\
&= a_0 + x\cdot \left(a_1 + a_2 x + a_3 x^2 + \dots + a_n x^{n-1}\right)\\
&= a_0 + x\cdot \left(a_1 + x\cdot\left( a_2 + a_3 x + \dots + a_n x^{n-2}\right)\right)\\
&\dots\\
&= a_0 + x\cdot \left(a_1 + x\cdot\left( a_2 + x\cdot \left(a_3 + \dots + x\cdot a_n\right)\right)\dots\right)
\end{aligned}
$$

Consider for example a fourth degree polynomial

$$
\begin{aligned}
p(x) &= a_0 + a_1x + a_2x^2 + a_3 x^3 + a_4x^4\\
&= a_0 + x\cdot \left(a_1 + x\cdot \left(a_2 + x \cdot \left(a_3 + x\cdot a_4\right)\right)\right)
\end{aligned}
$$

(counting the number of elops, at this point you will already observe that evaluating the polynomial in this way only requires 4 additions and 4 multiplications, which is less than what is required in the naive method: looking back at the example above, we needed 4 additions plus 10 multiplications).

Using the representation of the polynomial in the equation above, Horner's rule for evaluating a general polynomial $p(x)$ is given by the following algorithm:

### Algorithm: Polynomial evaluation with Horner's rule
**Input**: List $a=[a_0,a_1,\dots,a_n]$ of coefficients, value $x$ **Output**: Value of the polynomial $p(x)=\sum_{j=0}^{n} a_j\cdot x^j$
1. $p\mapsto a_n$
1. **for** $j=n-1,n-2,\dots,2,1,0$ **do**
1. $~~~~$ $p\mapsto a_j + x\cdot p$
1. **end for**
1. **return** $p$

Note that we are counting *down* from $n$ to $0$ in the for-loop and observe that we never have to explicitly raise $x$ to some power, so the code is also simpler than the pseudocode for the naive method.

Here is a Python implementation:

In [ ]:
def poly_horner(a_coeff,x):
    '''Evaluate polynomial p(x) = sum_{j=0}^{n} a_j*x^j with Horner's rule
    
    input:
      * a_coeff: List with polynomial cofficients a_0,a_1, ..., a_n
      * x: value at which to evaluate the polynomial
    output:
      value of polynomial p(x)
    '''
    degree = len(a_coeff)-1     # The list a_coeff contains n+1 elements, where n is the degree 
    poly = a_coeff[degree]    
    for j in range(degree-1,-1,-1):   # Loop from n down to 1
        poly = a_coeff[j] + x*poly # Update polynomial
    return poly

Let us double check that it works by evaluating the same third degree polynomial as above:

In [ ]:
a_coeff = [2,3,4,5]
x = 6
poly_horner(a_coeff,x)

### Efficiency of Horner's rule
How many elops does this algorithm require? Well, in each of the $n$ iterations we require 1 addition and 1 multiplication, so those are $n$ additions and $n$ multiplications in total and hence

$$
E_{\text{Horner}}(n) = 2n
$$

The following table compares the number of elops for the two methods and for different polynomial degrees $n$:

| problem size $$n$$ | $$E_{\text{naive}}(n)$$ | $$E_{\text{Horner}}(n)$$ | ratio $$E_{\text{naive}}(n)/E_{\text{Horner}}(n)$$ |
| -: | -: | -: | -: |
| 0 | 0 | 0 |1.5 |
| 1 | 2 | 2 |2.0 |
| 2 | 5 | 4 |2.5 |
| 3 | 9 | 6 |3.0 |
| 4 | 14 | 8 |3.5 |
| 5 | 20 | 10 |4.0 |
| 6 | 27 | 12 |4.5 |
| 7 | 35 | 14 |5.0 |
| 8 | 44 | 16 |5.5 |
| 9 | 54 | 18 |6.0 |
| 100 | 5150 | 200 |51.5 |
| 1000 | 501500 | 2000 |501.5 |

Horner's rule never requires more elops than the naive algorithm. As $n$ increases, it becomes increasingly more efficient than the naive method, for $n=1000$ we can reduce the number of iterations by a factor of just over 500 by using Horner's rule! Note that the improvement compared to the naive method is independent of the computer we use to run the code.

## Asymptotic growth of computational cost
We are usually interested in how the runtime grows as $n$ becomes very large. To quantify the rate of growth, one might look at how the runtime changes as the problem size doubles.

For Horner's rule this is straightforward. If we double $n$, the runtime increases by a factor two since

$$
T_\text{Horner}(2n) = \frac{E_{\text{Horner}}(2n)}{E_{\text{Horner}}(n)}\cdot T_{\text{Horner}}(n) = 2\cdot T_{\text{Horner}}(n)
$$

In other words, the runtime grows *linearly* with the problem size $n$. For the naive implementation we find

$$
T_\text{naive}(2n) = \frac{E_{\text{naive}}(2n)}{E_{\text{naive}}(n)}\cdot T_{\text{naive}}(n) = 
4 \cdot \underbrace{\frac{1+\frac{3}{2n}}{1+\frac{3}{n}}}_{=: \rho(n)}\cdot T_{\text{naive}}(n)
$$

It is easy to see that for large values of $n$ the factor $\rho(n)=\frac{1+\frac{3}{2n}}{1+\frac{3}{n}}$ is very close to $1$ (for example, $\rho(1000)\approx 0.9985$), so it makes sense to say

$$
T_\text{naive}(2n) \approx 4\cdot T_{\text{naive}}(n)
$$

In contrast to Horner's rule, the runtime of the naive algorithm increases by a factor of (approximately) four when the problem size $n$ doubles. Another way of seeing this is by noting that for large values of $n$ we can write

$$
E_{\text{naive}}(n) = \frac{1}{2}n^2\cdot \left(1+\frac{3}{2n}\right) \approx \frac{1}{2} n^2
$$

since $1+\frac{3}{2n} \rightarrow 1$ as $n\rightarrow \infty$. The number of elops is (approximately) proportional to the *square* of the problem size. In this case we say that computational cost grows *quadratically*.

We will come back to this next week, when we will develop mathematical tools to analyse the growth of the runtime more systematically.

# Check your understanding

Let $n!!$ be the double factorial that we met in one of the earlier lectures. How many elops are necessary to compute $n!!$ if $n$ is even?

1. $n$
2. $\frac{1}{2}n$
3. $\frac{1}{2}n-1$
4. $n^2-n$

To check your answer, enter it in the cell below and execute it

How many elops are required to compute the following sum:

$$
1 + 2 + 3! + 4! + 5! + \dots + n! = \sum_{j=1}^n j!
$$

for general $n\ge 2$?

(you can assume that we do not do any unnecessary multiplications with the factor $1$, i.e. we compute $3!=3\cdot 2$, $4!=4\cdot 3\cdot 2$ etc.)

1. $n$ elops
2. $\frac{1}{2}(n^2-n)$ elops
3. $n(n-1)$ elops
4. $n^3-2n$ elops

To check your answer, enter it in the cell below and execute it

Consider an algorithm which requires

$$
E(n) = \frac{3n^3+2n+7}{5}
$$

elops for a problem of size $n$. Assuming that $n$ is very large, which of the following statements is correct:

1. The runtime approximately doubles if $n$ increases by a factor two.
2. The runtime grows by a factor of approximately three if $n$ increases by a factor two.
3. The runtime approximately quadruples if $n$ increases by a factor two.
4. The runtime grows by a factor of approximately eight if $n$ increases by a factor two.

To check your answer, enter it in the cell below and execute it